# Data evaluation

In [5]:
import os
import json

In [6]:
experiments = [f"experiment_{i}" for i in range(1,18)]
DATA_PATH = "../data/"
ID_MAP_PATH = './movie_ids_map.json'
BASENAME_PATH = "./similar_movies.json"

THRESHOLD = 0.6
SAMPLE_SIZE = 100

In [7]:
def get_movie_ids_map():
    with open(ID_MAP_PATH, 'r') as f:
        movie_ids_map = json.load(f)
        
    return {int(k): v for k,v in movie_ids_map.items()}

In [8]:
def from_dict_of_lists_to_tuples(d):
    return [(int(k), v) for k, values in d.items() for v in values]

def get_baseline_data():
    with open(BASENAME_PATH, 'r') as f:
        baseline_similar_items = json.load(f)
    return baseline_similar_items

In [9]:
baseline_results = from_dict_of_lists_to_tuples(get_baseline_data())

In [10]:
def get_results_from_file(filename):
    with open(filename, 'r') as f:
        tuples_list = [eval(line.strip()) for line in f]
    return tuples_list

def get_parameters_from_file(filename):
    with open(filename, 'r') as f:
        parameters = json.load(f)
    return parameters

In [11]:
def get_pairs_over_threshold(l,t):
    filtered_tuples = filter(lambda r: r[1]>=t, l)
    return list(map(lambda r: r[0], list(filtered_tuples)))

def compute_confusion_matrix(baseline, experiment, n):
    baseline_set = set(baseline) 
    experiment_set = set(experiment)
    
    tp = len(baseline_set & experiment_set)  
    fp = len(experiment_set - baseline_set)  
    fn = len(baseline_set - experiment_set)  
    tn = n - tp - fp - fn
    
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

def compute_sensitivity_specificity(tp, fn, fp, tn):
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    return sensitivity, specificity

In [12]:
results_with_value = {}
parameters = {}
results_mapped = {}
for exp in experiments:
    filename = os.path.join(DATA_PATH,f'results_{exp}')
    results_with_value[exp] = get_results_from_file(filename)
    filename = os.path.join(DATA_PATH,f'parameters_{exp}.json')
    parameters[exp] = get_parameters_from_file(filename)
    id_map = get_movie_ids_map()
    results_mapped[exp] = list(map(lambda r: ((id_map[r[0][0]], id_map[r[0][1]]),r[1]), results_with_value[exp]))
    results_mapped[exp] += [((r[0][1], r[0][0]), r[1]) for r in results_mapped[exp]] #consider both orders of pairs
    if THRESHOLD:
        results_mapped[exp] = get_pairs_over_threshold(results_mapped[exp], THRESHOLD)

In [13]:
for exp in experiments:
    total_pairs = SAMPLE_SIZE*(SAMPLE_SIZE-1)
    conf = compute_confusion_matrix(baseline_results, results_mapped[exp], total_pairs)
    print("Experiment: ", exp)
    print("Parameters: ", parameters[exp])
    print(conf)
    print(compute_sensitivity_specificity(conf["TP"], conf["FN"], conf["FP"], conf["TN"]))
    print()

Experiment:  experiment_1
Parameters:  {'t_PCA': 0.95, 'l': 50, 't': 0.5, 'b': 17, 'h': 16}
{'TP': 56, 'FP': 140, 'FN': 2303, 'TN': 7401}
(0.02373887240356083, 0.9814348229677762)

Experiment:  experiment_2
Parameters:  {'t_PCA': 0.95, 'l': 50, 't': 0.5, 'b': 17, 'r': 3, 'h': 32}
{'TP': 56, 'FP': 138, 'FN': 2303, 'TN': 7403}
(0.02373887240356083, 0.9817000397825222)

Experiment:  experiment_3
Parameters:  {'t_PCA': 0.95, 'l': 50, 't': 0.5, 'b': 13, 'r': 4, 'h': 32}
{'TP': 42, 'FP': 152, 'FN': 2317, 'TN': 7389}
(0.017804154302670624, 0.9798435220792998)

Experiment:  experiment_4
Parameters:  {'t_PCA': 0.95, 'l': 50, 't': 0.5, 'b': 13, 'r': 4, 'h': 64}
{'TP': 50, 'FP': 138, 'FN': 2309, 'TN': 7403}
(0.0211954217888936, 0.9817000397825222)

Experiment:  experiment_5
Parameters:  {'t_PCA': 0.95, 'l': 50, 't': 0.5, 'b': 7, 'r': 8, 'h': 512}
{'TP': 19, 'FP': 63, 'FN': 2340, 'TN': 7478}
(0.008054260279779568, 0.9916456703354992)

Experiment:  experiment_6
Parameters:  {'t_PCA': 0.95, 'l': 50,